# Train a neural network rocket controller

Train a simple network to imitate the dataset's thrust and gimbal commands. Then submit it for simulation: a low prediction error alone does not guarantee a safe landing. The submitted model is scored on simulated landing performance, not agreement with the expert.

Place `train.ipynb`, `scaffold.py`, and `landings-varied.csv` together and run the notebook from that folder. Install the dependencies in the notebook's Python environment if needed: `python -m pip install torch safetensors numpy pandas matplotlib jupyter`.

The scaffold defines a small 12-32-2 network. It stores input normalization in the model, so the exported model accepts the same raw 12-value observations.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from safetensors.torch import save_file, load_file
from scaffold import FEATURES, TARGETS, create_model

landings_path = Path("landings-varied.csv")
seed = 42
learning_rate = 0.001
epochs = 30
batch_size = 256
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
landings = pd.read_csv(landings_path)
print(f"Loaded {len(landings):,} observations from {landings.run_id.nunique()} flights")
display(landings.head())
display(landings[FEATURES + TARGETS].describe())

## Split by flight

Observations from the same flight are strongly related. Keep whole `run_id` groups together so validation and test flights are unseen during training. The `success` column describes the final outcome and is not available as an input during a flight. You can experiment with filtering failed demonstrations using training data only.

In [ ]:
run_ids = np.random.default_rng(seed).permutation(landings.run_id.unique())
train_end = int(0.8 * len(run_ids))
val_end = int(0.9 * len(run_ids))
if train_end == 0 or val_end == train_end or val_end == len(run_ids):
    raise ValueError("The dataset needs enough distinct flights for train, validation and test splits.")
train = landings[landings.run_id.isin(run_ids[:train_end])].copy()
val = landings[landings.run_id.isin(run_ids[train_end:val_end])].copy()
test = landings[landings.run_id.isin(run_ids[val_end:])].copy()
for name, frame in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"{name}: {frame.run_id.nunique()} flights, {len(frame):,} observations")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, target in zip(axes, TARGETS):
    ax.hist(train[target], bins=40)
    ax.set(title=f"Training {target}", xlabel=target, ylabel="Observations")
plt.tight_layout()
plt.show()

## Prepare and train

Calculate input normalization using only the training flights, and store it inside the model. Pass **raw** observations to the model during training and inference. The network predicts both commands with mean squared error; sigmoid and tanh keep its outputs within the simulator's permitted ranges.


In [ ]:
mean = train[FEATURES].mean().to_numpy(dtype=np.float32)
std = train[FEATURES].std(ddof=0).to_numpy(dtype=np.float32)
std[std < 1e-8] = 1.0

def tensors(frame):
    x = frame[FEATURES].to_numpy(dtype=np.float32)  # Raw observations
    y = frame[TARGETS].to_numpy(dtype=np.float32)
    return torch.from_numpy(x), torch.from_numpy(y)

train_x, train_y = tensors(train)
val_x, val_y = tensors(val)
test_x, test_y = tensors(test)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)
model = create_model()
with torch.no_grad():
    model.mean.copy_(torch.from_numpy(mean))
    model.std.copy_(torch.from_numpy(std))
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()


In [ ]:
train_loss, val_loss = [], []
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    train_loss.append(total_loss / len(train_x))
    model.eval()
    with torch.no_grad():
        val_loss.append(loss_fn(model(val_x), val_y).item())
    print(f"Epoch {epoch + 1:3d}: train {train_loss[-1]:.5f}, validation {val_loss[-1]:.5f}")

In [ ]:
plt.plot(train_loss, label="Training")
plt.plot(val_loss, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.show()

## Evaluate and export

Use the held-out flights for a final prediction check. Then export the files for evaluation:

- Upload `model.py` and `weights.safetensors` to "Submit a PyTorch model". The weights include the training-set normalization.

The export copies your current scaffold to `model.py` so its architecture matches the trained weights. If you change the architecture, re-run training before exporting matching files.


In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(test_x).numpy()
errors = np.abs(predictions - test_y.numpy())
for i, target in enumerate(TARGETS):
    print(f"{target} test MAE: {errors[:, i].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (ax, target) in enumerate(zip(axes, TARGETS)):
    ax.scatter(test_y.numpy()[::10, i], predictions[::10, i], s=3, alpha=0.2)
    ax.set(xlabel="Expert", ylabel="Prediction", title=target)
plt.tight_layout()
plt.show()

In [ ]:
# Copy the exact scaffold used for this model without overwriting a different model.py.
source = Path("scaffold.py").read_bytes()
model_path = Path("model.py")
if model_path.exists() and model_path.read_bytes() != source:
    raise FileExistsError("model.py already exists and differs from scaffold.py. Move or rename it before exporting.")
if not model_path.exists():
    model_path.write_bytes(source)

model.eval()
state = {name: value.detach().cpu().contiguous() for name, value in model.state_dict().items()}
save_file(state, "weights.safetensors")

# Verify that a fresh copy of model.py loads the saved state and gives the same result.
spec = importlib.util.spec_from_file_location("exported_rocket_model", model_path)
exported_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exported_module)
reloaded = exported_module.create_model()
reloaded.load_state_dict(load_file("weights.safetensors"), strict=True)
reloaded.eval()
with torch.no_grad():
    sample = test_x[:1]
    torch.testing.assert_close(reloaded(sample), model(sample))
    expected = model(sample)[0].cpu().numpy()
print("Saved model.py and weights.safetensors for submission.")
print("Example action (thrust, gimbal):", tuple(expected))
